In [1]:
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_cohere import ChatCohere


load_dotenv()

True

In [2]:
import asyncio
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult,TextContent
from langchain_mcp_adapters.client import MultiServerMCPClient

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Interpret MCP tool calls: retry transient failures, surface all errors gracefully.

    -Retryable Mcp Error codes (e.g. -32603): retry with exponential backoff.
    -Non-Retryable Mcp error codes (e.g. -32602): return error message immediately.
    -Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on request.name"
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}) : {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(content=[TextContent(type="text", text=f"Tool Call failed (non-retryable) : {exc}")],
                                          isError=False,
                                          )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name}"
                      f"(attempt {attempt+1}/{self.max_retries}) : {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2**attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )


client = MultiServerMCPClient(
    {
        "travel_server" : {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
        }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [3]:
from typing import Dict,Any
from tavily import TavilyClient
from langchain.tools import tool


tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message" : "Search limit reached please summarize your findings and provide your final answer."}

    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error" : str(e)}

In [15]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database:  {e}"

In [5]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

In [6]:
from langchain.agents import create_agent

model = ChatCohere(model="command-r-08-2024", temperature=0)
travel_agent = create_agent(model,
                     tools=tools,
                     checkpointer=InMemorySaver(),
                     system_prompt="""You are a travel agent. Search for the flights to the desired destination wedding location.
                     You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
                     - Price (lowest, economy class)
                     - Duration (shortest)
                     - Date (time of the year which you believe is best for a wedding at this location)
                     - To make things easy, only look for one ticket, one way.
                     You may need to make multiple Searches to iteratively find the best options.
                     You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
                     If the MCP Tool fails, return malformed output, or does not give you usable flight results, try the tool again.
                     Once you have found the best options , let the user know your shortlist of options."""
                     )

In [7]:
from langchain.agents import create_agent

model = ChatCohere(model="command-r-08-2024", temperature=1.0)
venue_agent = create_agent(model,
                           tools=[web_search],
                           checkpointer=InMemorySaver(),
                           system_prompt="""You are a Venue Agent, Search for the Venues in the desired location and with the desired capacity.
                           You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
                           - Price (lowest)
                           - Capacity (exact match)
                           - Reviews (highest)
                           You may need to make multiple searches to iteratively find the best options.
                           You have suggested limit of 12 web Searches. Count every web search result you make.
                           After 12 Searches, you should stop searching and summarize the best options you have found so far.""")

In [8]:
from langchain.agents import create_agent

model = ChatCohere(model="command-r-08-2024", temperature=1.0)
music_agent = create_agent(model,
                           tools=[web_search],
                           checkpointer=InMemorySaver(),
                           system_prompt="""You are a Playlist Specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
                           Once you have a playlist, calculate the total duration and cost of the playlist, each song has an associated price.
                           If you run into errors when querying the database, try to fix them by making changes to the query.
                           Do not come back empty handed, keep trying to query the db until you find a list of songs.
                           This is a SQLite database. Before writing any data queries, first discover the schema.""")

In [9]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel Agent searches for the flights to the desired destination wedding location"""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]

    response = await travel_agent.ainvoke(
        {"messages" : [HumanMessage(content=f"Find flights from {origin} to {destination}")]}
    )

    return response["messages"][-1].content


@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity"""

    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"find venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({
        "messages" : [HumanMessage(content=query)]
    })
    return response["messages"][-1].content


@tool
def suggest_playlists(runtime: ToolRuntime)-> str:
    """Playlist agent curates the perfect playlist for the given genre"""
    genre = runtime.state["genre"]
    query = f"Find {genre} Tracks for wedding playlist"
    response = music_agent.invoke({"messages" : [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update state when you know all of the values: origin, destination, guest_count, genre.
    This tool must be called alone, without any other tool calls. It must be complete and return to make,
    the information available to other tools."""

    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages" : [ToolMessage("Successfully Updated state", tool_call_id = runtime.tool_call_id)]
    })

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
model = ChatCohere(model="command-r-08-2024", temperature=0)

coordinator = create_agent(model,
                     tools = [search_flights, search_venues, suggest_playlists, update_state],
                     state_schema=WeddingState,
                     checkpointer=memory,
                     system_prompt="""You are a Wedding Coordinator. First Find all the information you need to update the state. Once that has completed and returned, you can delegate the tasks to your Specialists for Flights, Venues and Music Playlists.
                     Once you have recieved the answer coordinate the Perfect Wedding for me.""")

In [16]:
from langchain.messages import HumanMessage


# while True:
#     user_input = input("Put your query Here or type quit to exit")
#     if user_input == "quit":
#         break
#     response = await coordinator.ainvoke({"messages" : [HumanMessage(content=user_input)]},
#                                    {"configurable" : {"thread_id" : "agent"}})
#     print(response["messages"][-1].content)

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="okay get them on work and provide me with the finally synthesized info")],
    },
    config={"configurable": {"thread_id": "wedding_1"}, "tags": ["WP"], "recursion_limit": 40}
)

In [17]:
from pprint import pprint

pprint(response["messages"][-1].content)

("I'm sorry, but I was unable to find a suitable jazz playlist for your "
 'wedding.\n'
 '\n'
 'However, I can provide you with some flight and venue options:\n'
 '\n'
 '## Flights\n'
 'The best time of year for a wedding in Paris is generally considered to be '
 'the summer months, from June to August. Here are some of the best flight '
 'options from India to Paris:\n'
 '\n'
 '| Price | Duration | Date |\n'
 '|---|---|---|\n'
 '| 769 EUR | 14 hours | December 22nd, 2026 |\n'
 '| 791 EUR | 14 hours and 20 minutes | December 30th, 2026 |\n'
 '| 796 EUR | 12 hours and 40 minutes | N/A |\n'
 '| 803 EUR | 9 hours and 30 minutes | N/A |\n'
 '\n'
 '## Venues\n'
 'Here are some venues in Paris that can accommodate 100 guests, listed in '
 'order of their average review score:\n'
 '\n'
 '1. Le Palais des Congrès Paris Saclay (4.3/5)\n'
 '2. The Peninsula Paris (5/5)\n'
 '3. Musée Rodin (4.92/5)\n'
 '4. Hotel Le Crillon (4/5)\n'
 '5. Ritz Paris (3.5/5)\n'
 '6. Mandarin Oriental Lutetia Paris (